# Four-model comparison: ridge vs attention decoders on Paul and three0

Visualizes the runs written by `run_four_model_comparison.py`. All four models predict the trial-averaged AIT natraster response (0-300 ms, 10 ms bins) from the same cached, mean-pooled DINOv3 ViT-L layers 3, 13, and 20:

| model | parameterization | fit |
|---|---|---|
| `ridge_site` | one linear map per (time bin, site) | closed form, LOO alpha per output |
| `ridge_timebin` | one linear map per time bin, all sites jointly | SGD on the Frobenius loss $\sum_t \lVert \hat Y_t - Y_t \rVert_F^2 + \sum_t \frac{\alpha_t}{n}\lVert W_t \rVert_F^2$ |
| `baseline` | `BaselineModel`: learned temporal queries cross-attend to DINOv3 features | AdamW, time-bin MSE |
| `site_token` | `SiteTokenBaselineModel`: query $q_{tn} = e_t + s_n$, one scalar per site | AdamW, time-bin MSE |
| `site_token_single` (optional) | the same model, trained on (image, site) pairs; the population is the stack of per-site passes | AdamW, 64 pairs per update |
| `site_token_matched` (optional) | the joint `site_token` with batch 3 and 117 epochs, matching `site_token_single`'s update count | AdamW, 75 targets per update |

The SGD models train for 100 epochs and keep the epoch with the best validation R2. Every score below is computed on the held-out test images of the chosen split(s). The notebook only scores models whose run file exists, so it can be rerun while the sweep is still going.

In [ ]:
import os
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml

# Locate the repository whether Jupyter starts in the project root or scripts folder.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [cwd, *cwd.parents] if (path / "config.yaml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate config.yaml from the current directory.")
# end if PROJECT_ROOT is None

ENV = os.getenv("MY_ENV", "tiziano_mac_mini")
with open(PROJECT_ROOT / "config.yaml", "r") as f:
    config = yaml.safe_load(f)
# end with project configuration
paths = config[ENV]["paths"]

# The run script defines the loaders, metrics, and model names; reuse them.
for source_path in (
    paths["useful_stuff_path"],
    str(PROJECT_ROOT / "python_scripts" / "src"),
    str(PROJECT_ROOT / "python_scripts" / "scripts"),
):
    if source_path not in sys.path:
        sys.path.insert(0, source_path)
    # end if source path is not registered
# end for source path

import run_four_model_comparison as comparison
# The run script selects the Agg backend for batch runs; restore inline plots.
%matplotlib inline
from IT_recap.dynamic_drsa import prepared_rdm_time_series
from IT_recap.neural_prediction_training import stimulus_correlation

In [ ]:
@dataclass
class Cfg:
    datasets: tuple = ("paul", "three0")
    seeds: tuple = (0,)
    results_root: Path = Path(paths["data_path"]) / "results" / "four_model_comparison"
    # Response window over which per-site scores are summarized (ms).
    summary_window_ms: tuple = (50.0, 300.0)
# EOC


cfg = Cfg()
# The sweep's own settings (time axis, RSA metrics, split fractions).
run_cfg = comparison.Cfg()
time_ms = run_cfg.time_start_ms + np.arange(30) * 1000.0 / run_cfg.target_fs

# Colorblind-validated categorical order; markers are the secondary encoding
# because the lines overlap.
MODEL_STYLE = {
    "ridge_site": {"color": "#2a78d6", "marker": "o"},
    "ridge_timebin": {"color": "#eb6834", "marker": "s"},
    "baseline": {"color": "#1baf7a", "marker": "^"},
    "site_token": {"color": "#eda100", "marker": "D"},
    "site_token_single": {"color": "#e87ba4", "marker": "v"},
    "site_token_matched": {"color": "#008300", "marker": "P"},
}
LABELS = comparison.MODEL_LABELS
cfg

## Load targets and score every finished run

For each split, targets are centered on the training-image mean per (time, site), exactly as in the sweep. Per time bin $t$:

- **R2**, pooled over sites: $R^2_t = 1 - \sum_{i,n}(\hat y_{itn}-y_{itn})^2 / \sum_{i,n} y_{itn}^2$
- **correlation**: Pearson $r$ across test images for every (t, site), averaged over sites
- **RSA**: Spearman correlation between the Euclidean RDMs of predicted and recorded population vectors at bin $t$ (the diagonal of the dRSA matrix)

In [ ]:
'''
score_runs
Score every finished model of one split on its test images.

INPUT:
    - targets: np.ndarray -> [images, time, sites] natraster responses
    - ann_index: np.ndarray -> ImageFolder index of every target row
    - run_dir: Path -> directory with seed<k>_<model>.npz files
    - seed: int -> split seed

OUTPUT:
    - scores: dict -> model -> per-bin metrics, per-cell correlations, run data
    - test_targets: np.ndarray -> [test images, time, sites] centered targets
'''
def score_runs(targets, ann_index, run_dir, seed):
    splits = comparison.make_image_splits(
        ann_index, seed, run_cfg.validation_fraction, run_cfg.test_fraction
    )
    test_targets = targets[splits["test"]] - targets[splits["train"]].mean(0)
    zero_mean = np.zeros(test_targets.shape[1:])
    neural_rdms = prepared_rdm_time_series(
        test_targets, run_cfg.drsa_rdm_metric, run_cfg.drsa_rsa_metric
    )

    scores = {}
    for name in comparison.MODELS:
        run_path = run_dir / f"seed{seed}_{name}.npz"
        if not run_path.is_file():
            continue
        # end if this model has not finished yet
        run = dict(np.load(run_path))
        predictions = run["test_predictions"]
        prediction_rdms = prepared_rdm_time_series(
            predictions, run_cfg.drsa_rdm_metric, run_cfg.drsa_rsa_metric
        )
        # [time, sites] Pearson r across test images.
        cell_correlations = stimulus_correlation(predictions, test_targets)
        # [time, sites] R2 of each cell against the fit-split mean.
        cell_r2 = 1.0 - ((predictions - test_targets) ** 2).sum(0) / (test_targets**2).sum(0)
        scores[name] = {
            "r2_by_bin": comparison.r2_by_timebin(predictions, test_targets, zero_mean),
            "corr_by_bin": np.nanmean(cell_correlations, axis=1),
            "rsa_by_bin": (neural_rdms * prediction_rdms).sum(axis=1),
            "r2": comparison.overall_r2(predictions, test_targets, zero_mean),
            "cell_correlations": cell_correlations,
            "cell_r2": cell_r2,
            "predictions": predictions,
            **{key: value for key, value in run.items() if key != "test_predictions"},
        }
    # end for model
    return scores, test_targets
# EOF


all_scores = {}    # dataset -> list over seeds of score dicts
all_targets = {}   # dataset -> list over seeds of centered test targets
for dataset_name in cfg.datasets:
    run_dir = cfg.results_root / dataset_name / "runs"
    if not run_dir.is_dir():
        print(f"{dataset_name}: no runs yet")
        continue
    # end if the dataset has not started
    targets, ann_index, _ = comparison.load_dataset(paths, dataset_name, run_cfg)
    all_scores[dataset_name], all_targets[dataset_name] = [], []
    for seed in cfg.seeds:
        scores, test_targets = score_runs(targets, ann_index, run_dir, seed)
        all_scores[dataset_name].append(scores)
        all_targets[dataset_name].append(test_targets)
    # end for split
    print(
        f"{dataset_name}: targets {targets.shape} | finished models "
        f"{list(all_scores[dataset_name][0])}"
    )
# end for dataset

## Summary table

Pooled test R2 and the per-bin correlation and RSA averaged over the response window (mean over splits if several).

In [ ]:
window = (time_ms >= cfg.summary_window_ms[0]) & (time_ms < cfg.summary_window_ms[1])
for dataset_name, seed_scores in all_scores.items():
    print(f"\n{dataset_name}  (correlation and RSA averaged over "
          f"{cfg.summary_window_ms[0]:.0f}-{cfg.summary_window_ms[1]:.0f} ms)")
    print(f"{'model':<34}{'params':>11}{'test R2':>9}{'corr':>8}{'RSA':>8}{'best ep':>9}")
    for name in seed_scores[0]:
        rows = [scores[name] for scores in seed_scores if name in scores]
        best_epoch = rows[0].get("best_epoch", None)
        print(
            f"{LABELS[name]:<34}{int(rows[0]['n_parameters']):>11,}"
            f"{np.mean([row['r2'] for row in rows]):>9.4f}"
            f"{np.mean([np.nanmean(row['corr_by_bin'][window]) for row in rows]):>8.4f}"
            f"{np.mean([np.nanmean(row['rsa_by_bin'][window]) for row in rows]):>8.4f}"
            f"{'' if best_epoch is None else int(best_epoch):>9}"
        )
    # end for model
# end for dataset

## Loss curves

Train (dashed) and validation (solid) MSE of the three SGD-trained models on the common, unpenalized MSE scale; the dotted line is the closed-form per-site ridge on the validation images. The marker shows the restored checkpoint. The right column shows the validation R2 used to pick it.

In [ ]:
trained_models = [name for name in comparison.MODELS if name != "ridge_site"]
figure, axes = plt.subplots(
    len(all_scores), 2, figsize=(14, 4.5 * len(all_scores)), squeeze=False
)
for row, (dataset_name, seed_scores) in enumerate(all_scores.items()):
    scores = seed_scores[0]  # curves of the first split
    loss_axis, r2_axis = axes[row]
    for name in trained_models:
        if name not in scores:
            continue
        # end if this model has not finished
        style = MODEL_STYLE[name]
        epochs = np.arange(1, len(scores[name]["validation_mse_curve"]) + 1)
        loss_axis.plot(epochs, scores[name]["validation_mse_curve"], "-",
                       color=style["color"], linewidth=2, label=LABELS[name])
        loss_axis.plot(epochs, scores[name]["train_mse_curve"], "--",
                       color=style["color"], linewidth=1.5)
        r2_axis.plot(epochs, scores[name]["validation_r2_curve"], "-",
                     color=style["color"], linewidth=2, label=LABELS[name])
        best_epoch = int(scores[name]["best_epoch"])
        if best_epoch > 0:
            loss_axis.plot(best_epoch, scores[name]["validation_mse_curve"][best_epoch - 1],
                           style["marker"], color=style["color"], markersize=9,
                           markeredgecolor="white")
            r2_axis.plot(best_epoch, scores[name]["validation_r2_curve"][best_epoch - 1],
                         style["marker"], color=style["color"], markersize=9,
                         markeredgecolor="white")
        # end if a trained checkpoint was restored
    # end for trained model
    if "ridge_site" in scores:
        loss_axis.axhline(scores["ridge_site"]["validation_mse"], color=MODEL_STYLE["ridge_site"]["color"],
                          linestyle=":", linewidth=2, label=f"{LABELS['ridge_site']} (validation)")
    # end if the closed-form reference exists
    loss_axis.set(xlabel="Epoch", ylabel="MSE (solid = validation, dashed = train)",
                  title=f"{dataset_name}: loss curves")
    r2_axis.set(xlabel="Epoch", ylabel="Validation R2 (pooled)",
                title=f"{dataset_name}: checkpoint selection criterion")
    for axis in (loss_axis, r2_axis):
        axis.grid(alpha=0.25)
        axis.legend(fontsize=8)
    # end for axis
# end for dataset
figure.tight_layout()
plt.show()

## Test R2, correlation, and RSA per time bin

Shaded bands are the SEM over splits when more than one split is loaded.

In [ ]:
panels = (
    ("r2_by_bin", "Test R2 (pooled over sites)"),
    ("corr_by_bin", "Mean Pearson r over sites"),
    ("rsa_by_bin", f"{run_cfg.drsa_rsa_metric} RDM correlation"),
)
figure, axes = plt.subplots(
    len(all_scores), 3, figsize=(19, 4.8 * len(all_scores)), squeeze=False
)
for row, (dataset_name, seed_scores) in enumerate(all_scores.items()):
    for axis, (key, ylabel) in zip(axes[row], panels):
        for name in comparison.MODELS:
            if name not in seed_scores[0]:
                continue
            # end if this model has not finished
            style = MODEL_STYLE[name]
            values = np.stack([scores[name][key] for scores in seed_scores])
            mean = np.nanmean(values, axis=0)
            axis.plot(time_ms, mean, color=style["color"], marker=style["marker"],
                      markersize=5, linewidth=2, label=LABELS[name])
            if len(values) > 1:
                sem = np.nanstd(values, axis=0) / np.sqrt(len(values))
                axis.fill_between(time_ms, mean - sem, mean + sem,
                                  color=style["color"], alpha=0.15, linewidth=0)
            # end if several splits give an SEM
        # end for model
        axis.axhline(0, color="gray", linewidth=0.8)
        axis.set(xlabel="Time from image onset (ms)", ylabel=ylabel,
                 title=f"{dataset_name}: {ylabel.lower()}")
        axis.grid(alpha=0.25)
        axis.legend(fontsize=8)
    # end for metric panel
# end for dataset
figure.tight_layout()
plt.show()

## Per-site comparison against per-site ridge

Each dot is one recording site: its test correlation averaged over the response window, for each model against the closed-form per-site ridge. Points above the diagonal are sites the model predicts better than ridge. This shows whether a gain is broad or carried by a few sites.

In [ ]:
# Every trained model that has finished on at least one dataset.
others = [
    name for name in comparison.MODELS
    if name != "ridge_site" and any(name in scores[0] for scores in all_scores.values())
]
figure, axes = plt.subplots(
    len(all_scores), len(others), figsize=(5.2 * len(others), 4.8 * len(all_scores)),
    squeeze=False,
)
for row, (dataset_name, seed_scores) in enumerate(all_scores.items()):
    scores = seed_scores[0]
    if "ridge_site" not in scores:
        continue
    # end if the reference has not finished
    # [sites] mean correlation over the response window.
    reference = np.nanmean(scores["ridge_site"]["cell_correlations"][window], axis=0)
    for axis, name in zip(axes[row], others):
        if name not in scores:
            axis.set_axis_off()
            continue
        # end if this model has not finished
        style = MODEL_STYLE[name]
        site_values = np.nanmean(scores[name]["cell_correlations"][window], axis=0)
        limits = (min(reference.min(), site_values.min()) - 0.02,
                  max(reference.max(), site_values.max()) + 0.02)
        axis.plot(limits, limits, color="gray", linewidth=1)
        axis.scatter(reference, site_values, s=60, color=style["color"],
                     marker=style["marker"], edgecolor="white", linewidth=1.5)
        wins = int((site_values > reference).sum())
        axis.set(xlim=limits, ylim=limits, xlabel=f"{LABELS['ridge_site']}: site r",
                 ylabel=f"{LABELS[name]}: site r",
                 title=f"{dataset_name}: {wins}/{len(reference)} sites above ridge")
        axis.grid(alpha=0.25)
    # end for compared model
# end for dataset
figure.tight_layout()
plt.show()

## Where in time and across sites the attention models differ from ridge

Difference in test correlation per (time bin, site) between each model and the per-site ridge. Sites are sorted by their ridge correlation over the response window (best at the top). Red means better than ridge, blue worse; the colour scale is symmetric around zero.

In [ ]:
figure, axes = plt.subplots(
    len(all_scores), len(others), figsize=(5.4 * len(others), 4.6 * len(all_scores)),
    squeeze=False,
)
for row, (dataset_name, seed_scores) in enumerate(all_scores.items()):
    scores = seed_scores[0]
    if "ridge_site" not in scores:
        continue
    # end if the reference has not finished
    reference_cells = scores["ridge_site"]["cell_correlations"]  # [time, sites]
    site_order = np.argsort(-np.nanmean(reference_cells[window], axis=0))
    differences = {
        name: (scores[name]["cell_correlations"] - reference_cells)[:, site_order]
        for name in others if name in scores
    }
    # Clip at the 98th percentile so a few noisy cells do not wash out the map.
    limit = np.nanpercentile(np.abs(np.stack(list(differences.values()))), 98)
    for axis, name in zip(axes[row], others):
        if name not in differences:
            axis.set_axis_off()
            continue
        # end if this model has not finished
        image = axis.imshow(
            differences[name].T, aspect="auto", cmap="RdBu_r", vmin=-limit, vmax=limit,
            extent=(time_ms[0], time_ms[-1] + 10, len(site_order), 0),
        )
        axis.set(xlabel="Time from image onset (ms)", ylabel="Site (sorted by ridge r)",
                 title=f"{dataset_name}: {LABELS[name]} - ridge")
        axis.yaxis.get_major_locator().set_params(integer=True)
        figure.colorbar(image, ax=axis, label="Delta r", extend="both")
    # end for compared model
# end for dataset
figure.tight_layout()
plt.show()

## Ridge regularization diagnostics

The penalties each ridge picked. `ridge_site` chooses one alpha per (time bin, site), `ridge_timebin` one per bin. An alpha at the top of the grid ($10^9$) means "predict the mean": expected before response onset, and a warning sign anywhere else.

In [ ]:
figure, axes = plt.subplots(
    len(all_scores), 2, figsize=(14, 4.2 * len(all_scores)), squeeze=False,
    gridspec_kw={"width_ratios": (1.4, 1)},
)
for row, (dataset_name, seed_scores) in enumerate(all_scores.items()):
    scores = seed_scores[0]
    heatmap_axis, line_axis = axes[row]
    if "ridge_site" in scores:
        image = heatmap_axis.imshow(
            np.log10(scores["ridge_site"]["alphas"]).T, aspect="auto", cmap="Blues",
            extent=(time_ms[0], time_ms[-1] + 10, scores["ridge_site"]["alphas"].shape[1], 0),
        )
        heatmap_axis.yaxis.get_major_locator().set_params(integer=True)
        figure.colorbar(image, ax=heatmap_axis, label="log10 alpha")
        heatmap_axis.set(xlabel="Time from image onset (ms)", ylabel="Site",
                         title=f"{dataset_name}: per-site ridge alpha")
        # Median over sites, to compare with the shared per-bin alpha.
        line_axis.plot(time_ms, np.log10(np.median(scores["ridge_site"]["alphas"], axis=1)),
                       color=MODEL_STYLE["ridge_site"]["color"], marker="o", linewidth=2,
                       label=f"{LABELS['ridge_site']} (median over sites)")
    # end if per-site ridge exists
    if "ridge_timebin" in scores:
        line_axis.plot(time_ms, np.log10(scores["ridge_timebin"]["alphas"]),
                       color=MODEL_STYLE["ridge_timebin"]["color"], marker="s", linewidth=2,
                       label=LABELS["ridge_timebin"])
    # end if per-bin ridge exists
    line_axis.set(xlabel="Time from image onset (ms)", ylabel="log10 alpha",
                  title=f"{dataset_name}: selected penalty per bin")
    line_axis.grid(alpha=0.25)
    line_axis.legend(fontsize=8)
# end for dataset
figure.tight_layout()
plt.show()

## Predicted vs recorded time courses

Population-mean response (mean over sites) for a few test images, recorded in black and predicted by each model. This is the qualitative check that the models track the response shape, not just its across-image ranking.

In [ ]:
n_example_images = 4
figure, axes = plt.subplots(
    len(all_scores), n_example_images, figsize=(4.5 * n_example_images, 3.6 * len(all_scores)),
    squeeze=False, sharey="row",
)
for row, (dataset_name, seed_scores) in enumerate(all_scores.items()):
    scores, test_targets = seed_scores[0], all_targets[dataset_name][0]
    # Pick the test images with the largest recorded population response.
    example_images = np.argsort(-np.abs(test_targets[:, window].mean(axis=(1, 2))))[:n_example_images]
    for axis, image_index in zip(axes[row], example_images):
        axis.plot(time_ms, test_targets[image_index].mean(-1), color="black",
                  linewidth=2.5, label="recorded")
        for name in comparison.MODELS:
            if name not in scores:
                continue
            # end if this model has not finished
            axis.plot(time_ms, scores[name]["predictions"][image_index].mean(-1),
                      color=MODEL_STYLE[name]["color"], marker=MODEL_STYLE[name]["marker"],
                      markersize=4, linewidth=1.5, label=LABELS[name])
        # end for model
        axis.set(xlabel="Time from image onset (ms)", title=f"{dataset_name}: test image {image_index}")
        axis.grid(alpha=0.25)
    # end for example image
    axes[row, 0].set_ylabel("Centered response (mean over sites)")
    axes[row, 0].legend(fontsize=7)
# end for dataset
figure.tight_layout()
plt.show()